# DePlag Brutal 95 Debug Notebook

Purpose: maximize honest accuracy without hiding failure modes. This notebook uses stronger paraphrase sources, source-balanced splits, threshold optimization, and per-source error analysis. It does not claim 95% until the held-out test actually reaches it.

In [ ]:
# CELL 1 - Setup
!pip install -q transformers datasets accelerate scikit-learn pandas numpy sentencepiece protobuf

import os, re, json, math, random, subprocess
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    subprocess.run(['nvidia-smi'], check=False)

PROJECT = '/content/DePlag_Brutal_95' if os.path.exists('/content') else os.getcwd()
os.makedirs(PROJECT, exist_ok=True)
os.chdir(PROJECT)
for d in ['data', 'models', 'reports', 'evaluation']:
    os.makedirs(d, exist_ok=True)
print('cwd:', os.getcwd(), '| cuda:', torch.cuda.is_available())

In [ ]:
# CELL 2 - Brutal configuration

MODEL_NAME = 'cross-encoder/quora-roberta-base'

# Start fast. If clean, increase EPOCHS to 2 and MAX_TRAIN_TOTAL to 240000.
MAX_LENGTH = 160
EPOCHS = 3
LR = 1e-5
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

# Source-balanced caps. This prevents QQP from dominating and PAWS/STSB from owning test difficulty.
MAX_TRAIN_PER_SOURCE = 30000
MAX_VAL_PER_SOURCE = 2500
MAX_TEST_PER_SOURCE = 2500
MAX_TRAIN_TOTAL = 180000

# Core curated HF source configs. v1 is manageable; v3 is huge and should be avoided until baseline is solved.
LANGCACHE_DATASET = 'redis/langcache-sentencepairs-v1'
LANGCACHE_CONFIGS = ['mrpc', 'paws', 'qqp', 'stsb', 'sick', 'pit2015', 'parade', 'apt']

# Optional plagiarism/paraphrase datasets. Keep off initially if runtime is tight.
USE_JPWAHLE_ETPC = True
USE_WORTA_APTY = True

print({
    'model': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'epochs': EPOCHS,
    'langcache_configs': LANGCACHE_CONFIGS,
})

In [ ]:
# CELL 3 - Dataset helpers

from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split

def clean_text(x):
    if not isinstance(x, str): return ''
    return ' '.join(x.strip().split())

def norm_key(x):
    x = clean_text(x).lower()
    x = re.sub(r'[^a-z0-9\s]', '', x)
    return ' '.join(x.split())

def normalize_labels(s):
    m = pd.Series(s).map(lambda x: str(x).strip().lower())
    d = {
        '1':1, 'true':1, 'yes':1, 'duplicate':1, 'paraphrase':1, 'equivalent':1,
        'entailment':1, 'positive':1, 'plagiarism':1, 'plagiarized':1,
        '0':0, 'false':0, 'no':0, 'not_duplicate':0, 'non-paraphrase':0,
        'non_paraphrase':0, 'neutral':0, 'contradiction':0, 'negative':0,
    }
    out = m.map(d)
    if out.isna().any():
        out = out.fillna(pd.to_numeric(s, errors='coerce'))
    if out.isna().any():
        raise ValueError('Unknown labels: ' + str(sorted(set(m[out.isna()].dropna()))[:10]))
    return out.astype(int)

def infer_pair_df(ds, source):
    df = ds.to_pandas() if hasattr(ds, 'to_pandas') else pd.DataFrame(ds)
    lower = {c.lower(): c for c in df.columns}
    pairs = [
        ('text_a','text_b'), ('sentence1','sentence2'), ('sent1','sent2'),
        ('question1','question2'), ('s1','s2'), ('source','target'),
        ('original','paraphrase'), ('original_text','paraphrased_text'),
        ('text','paraphrase')
    ]
    a = b = None
    for x, y in pairs:
        if x in lower and y in lower:
            a, b = lower[x], lower[y]
            break
    if a is None:
        strings = [c for c in df.columns if df[c].dtype == object]
        if len(strings) >= 2: a, b = strings[0], strings[1]
    label_col = next((lower[x] for x in ['label','labels','is_duplicate','duplicate','paraphrase','gold_label'] if x in lower), None)
    if a is None or label_col is None:
        raise ValueError(f'Cannot infer schema for {source}: {list(df.columns)}')
    out = pd.DataFrame({'text_a': df[a], 'text_b': df[b], 'label': normalize_labels(df[label_col])})
    out['text_a'] = out.text_a.map(clean_text); out['text_b'] = out.text_b.map(clean_text)
    out = out[(out.text_a != '') & (out.text_b != '') & out.label.isin([0,1])].copy()
    out['source'] = source
    return out[['text_a','text_b','label','source']].reset_index(drop=True)

def cap_balanced(df, max_n, name):
    if len(df) == 0: return df
    counts = df.label.value_counts().to_dict()
    if 0 not in counts or 1 not in counts:
        print('skip one-class split', name, counts)
        return pd.DataFrame(columns=df.columns)
    per = min(counts[0], counts[1], max_n // 2)
    return pd.concat([
        df[df.label == 0].sample(per, random_state=SEED),
        df[df.label == 1].sample(per, random_state=SEED),
    ]).sample(frac=1, random_state=SEED).reset_index(drop=True)

def add_key(df):
    x = df.text_a.map(norm_key); y = df.text_b.map(norm_key)
    df = df.copy(); df['pair_key'] = np.minimum(x, y) + ' ||| ' + np.maximum(x, y)
    return df.drop_duplicates('pair_key').reset_index(drop=True)

def split_df(df, src):
    df = add_key(df)
    trv, te = train_test_split(df, test_size=0.12, stratify=df.label, random_state=SEED)
    tr, va = train_test_split(trv, test_size=0.12, stratify=trv.label, random_state=SEED)
    return (
        cap_balanced(tr, MAX_TRAIN_PER_SOURCE, src + '_train'),
        cap_balanced(va, MAX_VAL_PER_SOURCE, src + '_val'),
        cap_balanced(te, MAX_TEST_PER_SOURCE, src + '_test'),
    )

In [ ]:
# CELL 4 - Load brutal dataset pool

train_parts, val_parts, test_parts, failed = [], [], [], []

for cfg in LANGCACHE_CONFIGS:
    try:
        ds = load_dataset(LANGCACHE_DATASET, cfg)
        if {'train','validation','test'}.issubset(ds.keys()):
            tr = cap_balanced(infer_pair_df(ds['train'], cfg), MAX_TRAIN_PER_SOURCE, cfg+'_train')
            va = cap_balanced(infer_pair_df(ds['validation'], cfg), MAX_VAL_PER_SOURCE, cfg+'_val')
            te = cap_balanced(infer_pair_df(ds['test'], cfg), MAX_TEST_PER_SOURCE, cfg+'_test')
        else:
            split = 'train' if 'train' in ds else list(ds.keys())[0]
            tr, va, te = split_df(infer_pair_df(ds[split], cfg), cfg)
        train_parts.append(tr); val_parts.append(va); test_parts.append(te)
        print('loaded', cfg, {'train':len(tr), 'val':len(va), 'test':len(te)})
    except Exception as e:
        failed.append((cfg, repr(e)))
        print('FAILED', cfg, repr(e))

if USE_JPWAHLE_ETPC:
    try:
        ds = load_dataset('jpwahle/etpc')
        split = 'train' if 'train' in ds else list(ds.keys())[0]
        tr, va, te = split_df(infer_pair_df(ds[split], 'etpc'), 'etpc')
        train_parts.append(tr); val_parts.append(va); test_parts.append(te)
        print('loaded etpc', {'train':len(tr), 'val':len(va), 'test':len(te)})
    except Exception as e:
        failed.append(('etpc', repr(e))); print('FAILED etpc', repr(e))

if USE_WORTA_APTY:
    try:
        ds = load_dataset('worta/apty')
        split = 'train' if 'train' in ds else list(ds.keys())[0]
        tr, va, te = split_df(infer_pair_df(ds[split], 'apty'), 'apty')
        train_parts.append(tr); val_parts.append(va); test_parts.append(te)
        print('loaded apty', {'train':len(tr), 'val':len(va), 'test':len(te)})
    except Exception as e:
        failed.append(('apty', repr(e))); print('FAILED apty', repr(e))

train_df = pd.concat([x for x in train_parts if len(x)], ignore_index=True)
val_df = pd.concat([x for x in val_parts if len(x)], ignore_index=True)
test_df = pd.concat([x for x in test_parts if len(x)], ignore_index=True)

# Global dedupe and leakage removal.
train_df = add_key(train_df).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = add_key(val_df); test_df = add_key(test_df)
train_keys = set(train_df.pair_key)
val_df = val_df[~val_df.pair_key.isin(train_keys)].reset_index(drop=True)
val_keys = set(val_df.pair_key)
test_df = test_df[~test_df.pair_key.isin(train_keys | val_keys)].reset_index(drop=True)

train_df = cap_balanced(train_df, MAX_TRAIN_TOTAL, 'global_train')
val_df = val_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print('\nFAILED:', failed)
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print('\n', name, len(d))
    print(d.source.value_counts())
    print(d.label.value_counts(normalize=True))

train_df.drop(columns=['pair_key']).to_csv('data/train.csv', index=False)
val_df.drop(columns=['pair_key']).to_csv('data/val.csv', index=False)
test_df.drop(columns=['pair_key']).to_csv('data/test.csv', index=False)

In [ ]:
# CELL 5 - Tokenize

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def to_ds(df):
    return Dataset.from_pandas(df[['text_a','text_b','label','source']].reset_index(drop=True), preserve_index=False)

raw = DatasetDict({'train':to_ds(train_df), 'validation':to_ds(val_df), 'test':to_ds(test_df)})

def tok(batch):
    return tokenizer(batch['text_a'], batch['text_b'], truncation=True, max_length=MAX_LENGTH)

encoded = raw.map(tok, batched=True, remove_columns=['text_a','text_b','source'])
encoded = encoded.rename_column('label', 'labels')
print(encoded)

In [ ]:
# CELL 6 - Metrics

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, precision_recall_curve, classification_report, confusion_matrix

def logits_to_proba(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = np.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)
    if logits.ndim == 1 or logits.shape[1] == 1:
        z = np.clip(logits.reshape(-1), -50, 50)
        return 1.0 / (1.0 + np.exp(-z))
    # 2-class softmax
    logits = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(np.clip(logits, -50, 50))
    return e[:, 1] / np.clip(e.sum(axis=1), 1e-12, None)

def metric_dict(y, p, t=0.5):
    y = np.asarray(y).astype(int); p = np.asarray(p)
    pred = (p >= t).astype(int)
    return {
        'accuracy': float(accuracy_score(y, pred)),
        'precision': float(precision_score(y, pred, zero_division=0)),
        'recall': float(recall_score(y, pred, zero_division=0)),
        'f1': float(f1_score(y, pred, zero_division=0)),
        'mcc': float(matthews_corrcoef(y, pred)),
        'roc_auc': float(roc_auc_score(y, p)) if len(set(y)) == 2 else 0.0,
        'threshold': float(t),
    }

def compute_metrics(ep):
    logits, labels = ep
    proba = logits_to_proba(logits)
    if np.isnan(proba).any():
        print('WARNING: NaN in predicted probabilities — returning zeros')
        proba = np.nan_to_num(proba, nan=0.5)
    return metric_dict(labels, proba, 0.5)

def sweep(y, p):
    rows = []
    for t in np.linspace(0.05, 0.95, 181):
        rows.append(metric_dict(y, p, t))
    return pd.DataFrame(rows).sort_values(['accuracy', 'mcc'], ascending=False)

In [ ]:
# CELL 7 - Train 2-class paraphrase cross-encoder (FIXED)
#
# Key fix: num_labels=2 (not 1) so HuggingFace uses CrossEntropyLoss
# for BOTH training and evaluation. This eliminates the NaN eval loss
# caused by the MSELoss/integer-label mismatch.

from transformers import (AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments,
                          Trainer, EarlyStoppingCallback, TrainerCallback)

# ── Sanity check labels before training ──
train_labels = np.array(encoded['train']['labels'])
val_labels   = np.array(encoded['validation']['labels'])
print(f'Train labels: {np.unique(train_labels, return_counts=True)}')
print(f'Val   labels: {np.unique(val_labels, return_counts=True)}')
assert set(np.unique(train_labels)) == {0, 1}, f'Bad train labels: {np.unique(train_labels)}'
assert set(np.unique(val_labels))   == {0, 1}, f'Bad val labels: {np.unique(val_labels)}'

# ── Model: 2-class head → native CrossEntropyLoss ──
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True)
model.config.use_cache = False
collator = DataCollatorWithPadding(tokenizer=tokenizer)
steps = max(1, math.ceil(len(encoded['train']) / TRAIN_BATCH_SIZE))
warmup = int(steps * EPOCHS * WARMUP_RATIO)
major = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
use_bf16 = bool(torch.cuda.is_available() and major >= 8)
use_fp16 = bool(torch.cuda.is_available() and not use_bf16)
print({'steps_per_epoch': steps, 'warmup': warmup, 'fp16': use_fp16, 'bf16': use_bf16})

args = TrainingArguments(
    output_dir='models/brutal_cross_encoder', learning_rate=LR,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS, weight_decay=WEIGHT_DECAY, warmup_steps=warmup,
    max_grad_norm=1.0, optim='adamw_torch', eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='f1', greater_is_better=True,
    logging_steps=100, logging_nan_inf_filter=True, save_total_limit=2,
    fp16=use_fp16, bf16=use_bf16, report_to='none', seed=SEED, dataloader_num_workers=2
)

# ── NaN guard callback ──
class NaNGuard(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            if math.isnan(logs['loss']) or math.isinf(logs['loss']):
                print(f'FATAL: NaN/Inf loss at step {state.global_step} — halting')
                control.should_training_stop = True

# ── Standard Trainer (no custom loss needed with num_labels=2) ──
trainer = Trainer(
    model=model, args=args,
    train_dataset=encoded['train'], eval_dataset=encoded['validation'],
    processing_class=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2), NaNGuard()]
)
trainer.train()
trainer.save_model('models/best_brutal_cross_encoder')
tokenizer.save_pretrained('models/best_brutal_cross_encoder')

In [ ]:
# CELL 8 - Threshold tuning, source audit, save results

def predict(split):
    pr = trainer.predict(encoded[split])
    return np.asarray(pr.label_ids).astype(int), logits_to_proba(pr.predictions)

y_val, p_val = predict('validation')
y_test, p_test = predict('test')
val_sweep = sweep(y_val, p_val)
display(val_sweep.head(15))
best_t = float(val_sweep.iloc[0].threshold)
pred_test = (p_test >= best_t).astype(int)
test_metrics = metric_dict(y_test, p_test, best_t)
print('BEST_THRESHOLD', best_t)
print('TEST', json.dumps(test_metrics, indent=2))
print(classification_report(y_test, pred_test, target_names=['Not Plagiarism','Plagiarism'], zero_division=0))

print('\nPER-SOURCE TEST')
source_rows = []
for src in sorted(test_df.source.unique()):
    idx = test_df.source.values == src
    m = metric_dict(y_test[idx], p_test[idx], best_t)
    m['source'] = src; m['n'] = int(idx.sum())
    source_rows.append(m)
    print(src, m)

pd.DataFrame(source_rows).sort_values('accuracy').to_csv('reports/per_source_metrics.csv', index=False)
results = {'model': MODEL_NAME, 'threshold': best_t, 'test': test_metrics, 'per_source': source_rows}
with open('reports/results.json', 'w') as f: json.dump(results, f, indent=2)
pd.DataFrame({'label': y_test, 'proba': p_test, 'pred': pred_test, 'source': test_df.source.values}).to_csv('reports/test_predictions.csv', index=False)
print('Saved reports/results.json, reports/per_source_metrics.csv, reports/test_predictions.csv')